[Levy et al. 2015](https://doi.org/10.1038/nature14279) tested fitness measurements for a series of ≈33 clones of  *Sacharomyces cerevisiae*.\
We have digitized the data from Fig. 2d of the original paper and replot the comparision as a correlation.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
import os
import os.path
from os import path

## create export directory if necessary
## foldernames for output plots/lists produced in this notebook
import os
FIG_DIR = f'./figures/Levy_data/'
os.makedirs(FIG_DIR, exist_ok=True)
print("All  plots will be stored in: \n" + FIG_DIR)

In [ ]:
%run setup_aesthetics.py

## Load data

In [ ]:
# load data
df = pd.read_excel('./data/Levy2015/Levy2016_Section8.1_PairwiseCompetition_Data.xlsx', header = 1)
## show first entries
df.head(3)

In [ ]:
## format into two columns
df['s_pair'] = df['Fluoro s']
df['s_bulk'] = df['Seq s']


## convert to float
df = df.astype('float')

In [ ]:
## show converted entries
df.head(3)

## Plot correlation of fitness values

In [ ]:
## fit a slope through intercept zero
from scipy.optimize import curve_fit

x = df['s_pair'].values
y = df['s_bulk'].values

lin = lambda x, a: a * x 
slope0 = curve_fit(lin, x, y)[0][0]

print(slope0)

In [ ]:
from scipy.stats import linregress, pearsonr
x = df['s_pair'].values
y = df['s_bulk'].values
slope, intercept,_,_,_ = linregress(x,y)
print(slope)

In [ ]:
## plot as correlation
fig, ax = plt.subplots(figsize = (FIGHEIGHT_TRIPLET, FIGHEIGHT_TRIPLET))

x = df['s_pair'].values
y = df['s_bulk'].values

ax.scatter(x,y, rasterized = True, color = 'silver', marker = 'o') 

xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
# make symmetric around around zero
#xymin, xymax = np.min([xmin,ymin,-ymax, -xmax]), np.max([xmax,ymax, -xmin,-ymin])
# make diagonal
xymin, xymax = np.min([xmin,ymin]), np.max([xmax,ymax])
ax.set_xlim(xymin, xymax)
ax.set_ylim(xymin,xymax)

xvec = np.linspace(xymin,xymax, num = 30)
#ax.plot(xvec,xvec*slope+intercept)

ax.plot([xymin,xymax], [xymin,xymax], ls = '--', color = 'black')

ax.set_ylabel('mutant fitness in bulk competition')
ax.set_xlabel('mutant fitness in pairwise competition')
ax.set_title(f'Levy et al. 2015 (n = {df.shape[0]} clones)', loc = 'right')

#fig.tight_layout()
fig.savefig(FIG_DIR+ 'fitness_pairwise_vs_fitness_bulk.pdf',
            dpi = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

### Plot absolute error

In [ ]:
data = df

In [ ]:
## full dataset
data = df
x = data['s_pair']
y = data['s_bulk'] - data['s_pair']

r, p = pearsonr(x, y)
print(r)
print(p)

In [ ]:
from scipy.stats import linregress

In [ ]:
data = df

x = data['s_pair'].values
y = data['s_bulk'].values - data['s_pair'].values


In [ ]:
### plot absolute error
fig, ax = plt.subplots(figsize = (FIGHEIGHT_TRIPLET, FIGHEIGHT_TRIPLET))
x = data['s_pair']
y = data['s_bulk'] - data['s_pair']

# plot data
ax.scatter(x,y, rasterized = True, color = 'silver', marker = 'o') 
# plot regression line
slope, intercept, r, p, se = linregress(x, y,)
xvec = np.linspace(xymin,xymax, num = 30)
ax.plot(xvec, slope*xvec+intercept, color = "tab:red")
print("r", r, "p", p)

ymin, ymax = ax.get_ylim()
yabsmax = np.max(np.abs([ymin,ymax]))
ax.set_ylim(-yabsmax,yabsmax)
ax.set_xlim(xymin,xymax)


ax.axhline(0, ls = '--', color = 'black')

ax.set_ylabel('absolute error')
ax.set_xlabel('mutant fitness in pairwise competition')
ax.set_title(f'Levy et al. 2015 (n = {df.shape[0]} clones)', loc = 'right')


#fig.tight_layout()
fig.savefig(FIG_DIR + 'absolute_error_bulk_competition.pdf',\
             dpi = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

### Plot relative error

In [ ]:
### Plot histogram of pairwise fitness values

x = np.abs( df['s_pair'].values)
fig,ax = plt.subplots()
_ = ax.hist(x, bins = 20)

In [ ]:
## compute relative error

x = df['s_pair'].values
delta = df['s_bulk'].values - df['s_pair'].values
y = np.abs(np.divide(delta, x, where = x!= 0))

## remove entries with zero relative erro
is_zero = y == 0
y = y[~is_zero]
x = x[~is_zero]

In [ ]:
## calculate mean relative error 
is_not_neutral = np.abs(x) > 0.02
rel_error_mean = np.exp(np.log(y[is_not_neutral]).mean())

In [ ]:
### plot absolute error
fig, ax = plt.subplots(figsize = (FIGHEIGHT_TRIPLET, FIGHEIGHT_TRIPLET))


ax.scatter(x,y, rasterized = True, color = 'silver', marker = 'o') 
ax.scatter(x[is_not_neutral],y[is_not_neutral], rasterized = True, color = 'blue')

ax.set_xlim(xymin,xymax)
ax.set_yscale('log')
#ax.set_ylim(ymin=1e-3)

#ax.axhline(0, ls = '--', color = 'black')
ax.axhline(rel_error_mean)

ax.set_ylabel('relative error')
ax.set_xlabel('mutant fitness in pairwise competition')
ax.set_title(f'Levy et al. 2015 (n = {df.shape[0]} clones)', loc = 'right')


#fig.tight_layout()
fig.savefig(FIG_DIR + 'relative_error_bulk_competition_with_rel_error_trend.pdf',\
             dpi = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

In [ ]:
print(rel_error_mean)

In [ ]:
y[is_not_neutral]

### Plot correlation with relative error trend

In [ ]:

## plot as correlation
fig, ax = plt.subplots(figsize = (FIGHEIGHT_TRIPLET, FIGHEIGHT_TRIPLET))

x = df['s_pair'].values
y = df['s_bulk'].values

ax.scatter(x,y, rasterized = True, color = 'silver', marker = 'o')
ax.scatter(x[is_not_neutral],y[is_not_neutral], rasterized = True, color = 'blue')

xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
xymin, xymax = np.min([xmin,ymin]), np.max([xmax,ymax])
ax.set_xlim(xymin, xymax)
ax.set_ylim(xymin,xymax)

xvec = np.linspace(xymin,xymax, num = 30)
ax.plot(xvec,xvec*(1+rel_error_mean), color = 'tab:blue')
ax.plot(xvec,xvec*(1-rel_error_mean), color = 'tab:blue')

ax.plot([xymin,xymax], [xymin,xymax], ls = '--', color = 'black')

ax.set_ylabel('mutant fitness in bulk competition')
ax.set_xlabel('mutant fitness in pairwise competition')
ax.set_title(f'Levy et al. 2015 (n = {df.shape[0]} knockouts)', loc = 'right')

#fig.tight_layout()
fig.savefig(FIG_DIR+ 'fitness_pairwise_vs_fitness_bulk_with_rel_error_trend.pdf',
            dpi = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

### Plot absolute error with relative error trend

In [ ]:

### plot absolute error
fig, ax = plt.subplots(figsize = (FIGHEIGHT_TRIPLET, FIGHEIGHT_TRIPLET))

x = df['s_pair'].values
y = df['s_bulk'].values - df['s_pair'].values

ax.scatter(x,y, rasterized = True, color = 'silver', marker = 'o') 
ax.scatter(x[is_not_neutral],y[is_not_neutral], rasterized = True, color = 'blue')

ymin, ymax = ax.get_ylim()
yabsmax = np.max(np.abs([ymin,ymax]))
ax.set_ylim(-yabsmax,yabsmax)
ax.set_xlim(xymin,xymax)

#xvec = np.linspace(xymin,xymax, num = 30)
#ax.plot(xvec,xvec*(rel_error_mean))

ax.axhline(0, ls = '--', color = 'black')

ax.set_ylabel('absolute error')
ax.set_xlabel('mutant fitness in pairwise competition')
ax.set_title(f'Levy et al. 2015 (n = {df.shape[0]} knockouts)', loc = 'right')


#fig.tight_layout()
fig.savefig(FIG_DIR + 'absolute_error_bulk_competition_with_rel_error_trend.pdf',\
             dpi = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

In [ ]:


## calculate correlation
from scipy.stats import pearsonr

r, p = pearsonr(x[is_not_neutral],y[is_not_neutral])
print(r)
print(p)

### Fit a correlation to residuals

In [ ]:
from scipy.stats import pearsonr

In [ ]:
## full dataset
data = df
x = data['s_pair']
y = data['s_bulk'] - data['s_pair']

r, p = pearsonr(x, y)
print(r)
print(p)

In [ ]:
## define outlier
index_outlier = y.idxmax()
is_outlier = np.array([v in [index_outlier] for v in df.index])

In [ ]:
## with outlier removed
data = df[~is_outlier]
x = data['s_pair'].values
y = data['s_bulk'].values - data['s_pair'].values

r, p = pearsonr(x, y)
print(r)
print(p)

### Fit a linear regression to residuals

In [ ]:
from scipy.stats import linregress

In [ ]:
data = df[~is_outlier]

x = data['s_pair'].values
y = data['s_bulk'].values - data['s_pair'].values
slope, intercept, r, p, se = linregress(x, y,)

In [ ]:
xvec = np.linspace(xymin,xymax, num = 30)